# Deep Learning Model - Home Credit Default Risk

This notebook is the report-facing entry point. The reusable implementation is in `deep_learning.py`; keeping the training code in a module makes it testable and avoids hidden notebook state.

## Architecture and preprocessing

The current model accepts one matrix containing standardized numerical values and categorical codes, and returns one logit. It uses fold-local median imputation, clipping and scaling, learned categorical embeddings, a custom trainable `FeatureGate`, a custom `CrossNetwork` and a residual deep tower. An alternative single-input model is in `model_transformer.py`.

Regularization consists of LayerNorm, dropout, decoupled weight decay, gradient clipping, early stopping, and a ReduceLROnPlateau scheduler. The optimizer `DecoupledAdamW` is implemented directly from `torch.optim.Optimizer`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
from deep_learning import Config, CreditRiskNet, FeatureGate, CrossNetwork, DecoupledAdamW, run

## Reproduce training

The current configuration uses all training rows, 5 folds and up to 30 epochs with early stopping after 5 epochs without ROC-AUC improvement. Set `tune_trials=3` to run the optional holdout parameter search before cross-validation.

In [ ]:
# Uncomment for the full reproducible run (can take a long time on CPU).
# config = Config(folds=5, epochs=30, patience=5, output_dir='artifacts/deep_cross_30_epochs')
# summary = run(config, tune_trials=3, tuning_rows=60000)

## Collected results

In [ ]:
GPU_RUNS = {name: Path(f'artifacts/{name}_30_epochs_gpu') for name in ('deep_cross', 'transformer')}
comparison = pd.DataFrame([{'model': name, **{key: json.loads((path / 'metrics_summary.json').read_text())[key] for key in ('local_roc_auc', 'local_pr_auc', 'std_fold_roc_auc', 'elapsed_minutes')}} for name, path in GPU_RUNS.items()])
comparison.sort_values('local_roc_auc', ascending=False)

In [ ]:
ARTIFACTS = Path('artifacts/deep_cross_30_epochs_gpu')
summary = json.loads((ARTIFACTS / 'metrics_summary.json').read_text())
pd.Series(summary).drop('config').to_frame('value')

In [ ]:
display(Image(filename=str(ARTIFACTS / 'training_curves.png')))
display(Image(filename=str(ARTIFACTS / 'gradient_flow.png')))

In [ ]:
display(pd.read_csv(ARTIFACTS / 'fold_metrics.csv'))
if (ARTIFACTS / 'tuning_log.csv').exists():
    display(pd.read_csv(ARTIFACTS / 'tuning_log.csv'))

## Leaderboard comparison and adequacy

Upload a GPU model submission from its artifact directory to Kaggle, then enter the returned Public score in `leaderboard_results.csv`. The Private score only becomes available when the competition reveals it. The full GPU model comparison is in `artifacts/GPU_COMPARISON.md`.

Adequacy should be judged against the LightGBM local baseline (ROC-AUC 0.7639, PR-AUC 0.2479), fold stability, and the Public/Private gap. The model is adequate as a neural baseline if it clearly beats random ranking, trains without dead/exploding gradients, and its validation-to-leaderboard gap is small. It is competitive only if it approaches or exceeds the tree baseline; otherwise the tabular inductive bias of boosting remains stronger and an ensemble is the sensible next step.

In [ ]:
leaderboard = pd.read_csv(ARTIFACTS / 'leaderboard_results.csv')
leaderboard